Encoder:  
- Word Vocab 
- Normal Encode


-> Hidden_state -> Decoder 

Decoder:
- Phoneme Vocab 
- 4 embedding layer + Concat 4 layer - Decoder layer
- Decoder output: (B, S, hidden_size)
- 4 Feed-Forward Layers
- ViPhoneme decode 



`viword_vocab` methods: 
- `make_vocab`
- `encode_caption`
- `decode_caption`
- `decode_batch_caption`


Shapes: 
- `input:` (B, S, 3)
- `bos input:` (bos_idx, pad_idx, pad_idx)
- `eos input:` (eos_idx, pad_idx, pad_idx)


`TextSumDatasetPhoneme` methods: 
- `make_vocab`
- `encode_caption`
- `decode_caption`
- `decode_batch_caption`

Shapes: 
- `input_ids`: (bos_idx, pad_idx, pad_idx) + (onset, middle, lower) + (eos_idx, pad_idx, pad_idx)
- `label`: (bos_idx, pad_idx, pad_idx) + (onset, middle, lower) + (eos_idx, pad_idx, pad_idx)
- `shifted_right_label`: (onset, middle, lower) + (eos_idx, pad_idx, pad_idx)

In [10]:
import torch 

In [11]:
data = [20, 15, 40]

In [12]:
data = torch.Tensor(data)

In [16]:
import torch

data = [
    [
        [1, 2, 3],
        [4, 5, 6]
    ],
    [
        [7, 8, 9],
        [10, 11, 12]
    ]
]

data = torch.Tensor(data)

print(data)
print(data.shape)

tensor([[[ 1.,  2.,  3.],
         [ 4.,  5.,  6.]],

        [[ 7.,  8.,  9.],
         [10., 11., 12.]]])
torch.Size([2, 2, 3])


In [24]:
data.size()[-1]

3

In [26]:
from torch import nn

class FeedForward(nn.Module):
    def __init__(self, input: torch.Tensor):
        super(FeedForward, self).__init__()
        self.hidden_size = input.size()[-1]
        self.norm = nn.LayerNorm(self.hidden_size)
        self.linear_projection_1 = nn.Linear(self.hidden_size, \
                                            self.hidden_size * 2)
        self.ReLu = nn.ReLU()
        self.linear_projection_2 = nn.Linear(self.hidden_size*2,\
                                            self.hidden_size)

    def forward(self, x):
        x = self.norm(x)
        x_rest = x
        x = self.ReLu(self.linear_projection_1(x))
        x = self.linear_projection_2(x)
        x += x_rest 
        return x
    
ff = FeedForward(data)
x = ff.forward(data)

In [30]:
x

tensor([[[-1.0705,  0.4015,  1.2111],
         [-1.0705,  0.4015,  1.2111]],

        [[-1.0705,  0.4015,  1.2111],
         [-1.0705,  0.4015,  1.2111]]], grad_fn=<AsStridedBackward0>)

In [31]:
data

tensor([[[ 1.,  2.,  3.],
         [ 4.,  5.,  6.]],

        [[ 7.,  8.,  9.],
         [10., 11., 12.]]])

In [32]:
import torch

a = torch.tensor([[1,2]])
b = torch.tensor([[3,4]])

print(torch.cat([a,b], dim=0))
print(torch.concat([a,b], dim=0))

tensor([[1, 2],
        [3, 4]])
tensor([[1, 2],
        [3, 4]])


In [33]:
import torch

batch_size = 4
hidden_size = 128

x = torch.randn(batch_size, 1, hidden_size * 3)

print(x.shape)

torch.Size([4, 1, 384])


In [2]:
total = [3,4,5,6]
test = sum(total)
test

18

In [4]:
import torch
x = torch.tensor([[5]])
print(x.item())

5


In [5]:
data = [[4, 5, 6], [5, 6, 7]]

In [7]:
len(data[0])

3

In [8]:
import os
print(os.cpu_count())

16


In [12]:
"".startswith("w")

False

In [13]:
import re
import unicodedata

def get_tone(word: str):
    tone_map = {
        '\u0300': '<`>',
        '\u0301': '</>',
        '\u0303': '<~>',
        '\u0309': '<?>',
        '\u0323': '<.>',
    }
    decomposed_word = unicodedata.normalize('NFD', word)
    tone = None
    remaining_word = ''
    for char in decomposed_word:
        if char in tone_map:
            tone = tone_map[char]
        else:
            remaining_word += char
    remaining_word = unicodedata.normalize('NFC', remaining_word)

    return tone, remaining_word


def get_onset(word: str) -> tuple[str, str]:
    onsets = ['ngh', 'tr', 'th', 'ph', 'nh', 'ng', 'kh',
              'gi', 'gh', 'ch', 'q', 'đ', 'x', 'v', 't',
              's', 'r', 'n', 'm', 'l', 'k', 'h', 'g', 'd',
              'c', 'b']

    # get the onset
    for onset in onsets:
        if word.startswith(onset):
            if onset != "q":
                word = word.removeprefix(onset)
            return onset, word

    return None, word


def get_medial(word: str) -> tuple[str, str]:
    O_MEDIAL = "o"
    U_MEDIAL = "u"

    if word.startswith("q"):
        # in Vietnamese, words starting with "q" always has "u" as the medial
        word = word.removeprefix("qu")
        return U_MEDIAL, word

    o_medial_cases = ["oa", "oă", "oe"]
    for o_medial_case in o_medial_cases:
        if word.startswith(o_medial_case):
            word = word.removeprefix("o")
            return O_MEDIAL, word

    if word.startswith("ua") or word.startswith("uô"):
        return None, word

    nucleuses = ['ê', 'y', 'ơ', 'a', 'â', 'ya']
    for nucleus in nucleuses:
        component = U_MEDIAL + nucleus
        if word.startswith(component):
            word = word.removeprefix("u")
            return U_MEDIAL, word

    return None, word


def get_nucleus(word: str) -> tuple[str, str]:
    nucleuses = ['oo', 'ươ', 'ưa', 'uô', 'ua', 'iê', 'yê',
                 'ia', 'ya', 'e', 'ê', 'u', 'ư', 'ô', 'i',
                 'y', 'o', 'ơ', 'â', 'a', 'o', 'ă']

    for nucleus in nucleuses:
        if word.startswith(nucleus):
            word = word.removeprefix(nucleus)
            return nucleus, word

    return None, word


def get_coda(word: str) -> str:
    codas = ['ng', 'nh', 'ch', 'u', 'n', 'o',
             'p', 'c', 'm', 'y', 'i', 't', 'k']

    if word in codas:
        return word

    return None


def split_phoneme(word: str) -> list[str, str, str]:
    onset, word = get_onset(word)

    medial, word = get_medial(word)

    nucleus, word = get_nucleus(word)

    coda = get_coda(word)

    return onset, medial, nucleus, coda


def is_Vietnamese(word: str) -> tuple[bool, tuple]:
    tone, word = get_tone(word)
    if not re.match(r"[a-zA-Zăâđưôơê]", word):
        return False, None

    # handling for special cases
    special_words_to_words = {
        "gin": "giin",     # gìn after being removed the tone
        "giêng": "giiêng",  # giếng after being removed the tone
        "giêt": "giiêt",   # giết after being removed the tone
        "giêc": "giiêc",   # giếc (diếc) after being removed the tone
        "gi": "gii"        # gì after removing the tone
    }

    if word in special_words_to_words:
        word = special_words_to_words[word]

    # check the total number of nucleus in word
    vowels = ['oo', 'ươ', 'ưa', 'uô', 'ua', 'iê', 'yê',
              'ia', 'ya', 'e', 'ê', 'u', 'ư', 'ô', 'i',
              'y', 'o', 'ơ', 'â', 'a', 'o', 'ă']
    currentCharacterIsVowels = False
    previousCharacterIsVowels = word[0] in vowels
    foundVowels = 0

    for character in word[1:]:
        if character in vowels:
            currentCharacterIsVowels = True
        else:
            currentCharacterIsVowels = False

        if currentCharacterIsVowels and not previousCharacterIsVowels:
            foundVowels += 1

        # in Vietnamese, each word has only one syllable
        if foundVowels > 2:
            return False, None

        previousCharacterIsVowels = currentCharacterIsVowels

    # in case the word has the structure of a Vietnamese word, we check whether it satisfies the rule of phoneme combination
    onset, medial, nucleus, coda = split_phoneme(word)

    if nucleus is None:
        return False, None

    former_word = ""
    for component in [onset, medial, nucleus, coda]:
        if component is not None:
            former_word += component
    if former_word != word:
        return False, None

    if onset == "k" and medial is None and nucleus not in ["i", "y", "e", "ê", "iê", "yê", "ia", "ya"]:
        return False, None

    if onset == "c" and medial is None and nucleus in ["i", "y", "e", "ê", "iê", "yê", "ia", "ya"]:
        return False, None

    if onset == "q" and not medial == "u":
        return False, None

    if onset == "gh" and medial is None and nucleus not in ["i", "e", "ê", "iê"]:
        return False, None

    if onset == "g" and medial is None and nucleus in ["i", "e", "ê", "iê"]:
        return False, None

    if onset == "ngh" and medial is None and nucleus not in ["i", "e", "ê", "iê", "yê", "ia", "ya"]:
        return False, None

    if onset == "ng" and medial is None and nucleus in ["i", "e", "ê", "iê", "yê", "ia", "ya"]:
        return False, None

    if onset in ["r", "gi"] and medial is not None:
        return False, None

    if medial == "o" and nucleus not in ["a", "ă", "e"]:
        return False, None

    if medial == "u" and nucleus not in ['yê', 'ya', 'e', 'ê', 'y', 'ơ', "ô", 'a', 'â', 'ă']:
        return False, None

    if nucleus == "oo" and coda not in ["ng", "c"]:
        return False, None

    if nucleus == "ua" and coda is not None:
        return False, None

    if nucleus == "ia" and coda is not None:
        return False, None

    if nucleus == "ya" and coda is not None:
        return False, None

    if nucleus in ["ua", "uô"] and coda == "ph":
        return False, None

    if nucleus in ["yê", "iê"] and coda is None:
        return False, None

    if nucleus in ["ă", "â"] and coda is None:
        return False, None

    if medial == "o" and nucleus in ["iê", "yê", "ia", "ya"]:
        return False, None

    if medial is not None:
        if nucleus in ["u", "oo", "o", "ua", "uô", "ươ", "ưa", "ư"]:
            return False, None

        if nucleus in ["i", "e", "ê", "ia", "ya", "iê", "yê"] and coda in ["m", "ph"]:
            return False, None

    if coda == "o" and nucleus not in ["a", "e"]:
        return False, None

    if coda == "y" and nucleus not in ["a", "â"]:
        return False, None

    if coda == "i" and nucleus in ["ă", "â", "i", "e", "iê", "yê", "ia", "ya"]:
        return False, None

    if coda == "nh" and nucleus not in ["a", "i", "y", "ê"]:
        return False, None

    if coda == "ng" and nucleus not in ["a", "o", "ô", "u", "ư", "e", "iê", "ươ", "â", "ă", "uô", "oo"]:
        return False, None

    if coda == "ch" and nucleus not in ["i", "a", "ê", "y"]:
        return False, None

    if coda == "c" and nucleus in ["i", "ê", "e", "ơ"]:
        return False, None

    if nucleus == coda:
        return False, None

    return True, (onset, medial, nucleus, coda, tone)


# def compose_word(onset: str, medial: str, nucleus: str, coda: str, tone: str) -> str:
#     tone_map = {
#         '<`>': '\u0300',
#         '</>': '\u0301',
#         '<~>': '\u0303',
#         '<?>': '\u0309',
#         '<.>': '\u0323'
#     }
#     tone = tone_map[tone]

#     # process for the special case of medial + coda (hỏa, thủy, thuở, thỏa, ...)
#     # in this case, only "thuở" follows the general rule of tone marking, the others are the case that tones are marked on the medial.
#     if onset != "q" and medial is not None and nucleus is not None and coda is None and nucleus != "ơ":
#         medial += tone
#     else:
#         if coda is None:
#             nucleus = nucleus[0] + tone + nucleus[1:]
#         else:
#             nucleus = nucleus + tone

#     word = ""
#     if onset:
#         word += onset
#     if medial:
#         word += medial
#     if nucleus:
#         word += nucleus
#     if coda:
#         word += coda

#     if "gii" in word:
#         word = re.sub("gii", "gi", word)

#     return word

def compose_word(initial: str, rhyme: str, tone: str) -> tuple[str, None]:
    def get_glide(rhyme: str) -> str:
        if rhyme.startswith("w"):
            return "w"

        return None

    def get_final(rhyme: str) -> str:
        possible_finals = ['j', 'm', 'n', 'ŋ̟', 'ŋ', 'p', 'k', 't', 'k̟', 'w']
        for final in possible_finals:
            if rhyme.endswith(final):
                return final

        return None

    initial2_onset = {
        None: [None],
        "m": ["m"],
        "b": ["b"],
        "k": ["c", "k", "q"],
        "v": ["v"],
        "t": ["t"],
        "ʝ": ["d"],
        "d": ["đ"],
        "n": ["n"],
        "r": ["r"],
        "s": ["x"],
        "ʂ": ["s"],
        "l": ["l"],
        "h": ["h"],
        "f": ["ph"],
        "tʰ": ["th"],
        "ɣ": ["g", "gh"],
        "z": ["gi"],
        "t͡ɕ": ["ch"],
        "ʈ͡ʂ": ["tr"],
        "ɲ": ["nh"],
        "ŋ": ["ng", "ngh"],
        "χ": ["kh"],
    }

    glide2medial = {
        None: [None],
        "w": ["u", "o"],
    }

    vowel2nucleus = {
        None: [None],
        "iə": ["iê", "yê", "ia", "ya"],
        "uə": ["uô", "ua"],
        "ɯə": ["ươ", "ưa"],
        "a": ["a"],
        "ă": ["a", "ă"],
        "ə̆": ["â"],
        "i": ["i", "y"],
        "ɛ": ["e"],
        "e": ["ê"],
        "u": ["u"],
        "ɯ": ["ư"],
        "ɔ": ["o"],
        "ɔː": ["oo"],
        "o": ["ô"],
        "ə": ["ơ"],
    }

    final2coda = {
        None: [None],
        "j": ["i", "y"],
        "m": ["m"],
        "n": ["n"],
        "ŋ̟": ["nh"],
        "ŋ": ["ng"],
        "p": ["p"],
        "t": ["t"],
        "k": ["c"],
        "k̟": ["ch"],
        "w": ["u", "o"],
    }

    glide: str = get_glide(rhyme)
    final: str = get_final(rhyme)
    vowel = rhyme
    if glide is not None:
        vowel = vowel.removeprefix(glide)
    if final is not None:
        vowel = vowel.removesuffix(final)

    coda_options = final2coda[final]
    if len(coda_options) == 1:
        coda = coda_options[0]
    else:
        # special cases where we have multiple options for the coda
        if coda_options == ["i", "y"]:
            if vowel in ["ă", "ə̆"]:
                coda = "y"
            else:
                coda = "i"

        if coda_options == ["u", "o"]:
            if vowel in ["ă", "ə̆", "e", "i", "iə", "ɯ", "ɯə"]:
                coda = "u"
            else:
                coda = "o"

    nucleus_options = vowel2nucleus[vowel]
    if len(nucleus_options) == 1:
        nucleus = nucleus_options[0]

    onset_options = initial2_onset[initial]
    if len(onset_options) == 1:
        onset = onset_options[0]

    # special cases where we have multiple options for the nucleus
    if nucleus_options == ["iê", "yê", "ia", "ya"]:
        nucleus = None
        if glide is None and final is None:
            nucleus = "ia"
        if glide is not None and final is None:
            nucleus = "ya"
        if glide is None and final is not None:
            nucleus = "iê"
        if glide is not None and final is not None:
            nucleus = "yê"
        if initial is None and glide is None:
            nucleus = "yê"
        # in the case where the syllabic components are connot correctly given
        if nucleus is None:
            return None

    if nucleus_options == ["uô", "ua"]:
        nucleus = None
        if final is None:
            nucleus = "ua"
        else:
            nucleus = "uô"
        # in the case where the syllabic components are connot correctly given
        if nucleus is None:
            return None

    if nucleus_options == ["ươ", "ưa"]:
        nucleus = None
        if final is None:
            nucleus = "ưa"
        else:
            nucleus = "ươ"
        # in the case where the syllabic components are connot correctly given
        if nucleus is None:
            return None

    if nucleus_options == ["i", "y"]:
        nucleus = None
        if initial is None and glide is None and final is None:
            nucleus = "y"
        elif glide is not None:
            nucleus = "y"
        else:
            nucleus = "i"
        # in the case where the syllabic components are connot correctly given
        if nucleus is None:
            return None

    if nucleus_options == ["a", "ă"]:
        if coda in ["u", "y"]:
            nucleus = "a"
        else:
            nucleus = "ă"

    if onset_options == ["c", "k", "q"]:
        if glide:
            onset = "q"
        else:
            if nucleus in ["i", "y", "e", "ê", "iê", "yê", "ia", "ya"]:
                onset = "k"
            else:
                onset = "c"

    if onset_options == ["ng", "ngh"]:
        if nucleus in ["i", "e", "ê", "iê", "yê", "ia", "ya"] and glide is None:
            onset = "ngh"
        else:
            onset = "ng"

    if onset_options == ["g", "gh"]:
        if nucleus in ["i", "e", "ê", "iê"]:
            onset = "gh"
        else:
            onset = "g"

    medial_options = glide2medial[glide]
    if len(medial_options) == 1:
        medial = medial_options[0]
    else:
        if onset == "q":
            medial = "u"
        else:
            # special cases where we have multiple options for the medial
            if nucleus in ["e", "a", "ă"]:
                medial = "o"
            else:
                medial = "u"

    tone_map = {
        "-": '',
        '˨˩': '\u0300',
        '˧˥': '\u0301',
        '˧ˀ˥': '\u0303',
        '˧˩': '\u0309',
        '˧ˀ˩': '\u0323'
    }
    tone = tone_map[tone]
    # process for the special case of medial + coda (hỏa, thủy, thuở, thỏa, ...)
    # in this case, only thuở, nhuệ/huệ/huề/duệ follows the general rule of tone marking, the others are the case that tones are marked on the medial.
    if onset != "q" and medial is not None and nucleus is not None and coda is None and nucleus not in ["ơ", "ê"]:
        medial += tone
    else:
        if coda is None:
            nucleus = nucleus[0] + tone + nucleus[1:]
        else:
            nucleus = nucleus + tone

    word = ""
    if onset:
        word += onset
    if medial:
        word += medial
    if nucleus:
        word += nucleus
    if coda:
        word += coda

    if "gii" in word:
        word = re.sub("gii", "gi", word)

    word = unicodedata.normalize('NFC', word)

    return word

def convert_Vietnamese_to_IPA(syllable: str) -> list[str]:
    assert re.search(r"\s+", syllable) is None, "The input must be a syllable"

    onset2ipa = {
        None: None,
        "m": "m",
        "b": "b",
        "c": "k",
        "k": "k",
        "v": "v",
        "t": "t",
        "d": "ʝ",
        "đ": "d",
        "n": "n",
        "r": "r",
        "x": "s",
        "s": "ʂ",
        "l": "l",
        "h": "h",
        "ph": "f",
        "th": "tʰ",
        "g": "ɣ",
        "gh": "ɣ",
        "gi": "z",
        "ch": "t͡ɕ",
        "tr": "ʈ͡ʂ",
        "nh": "ɲ",
        "ng": "ŋ",
        "ngh": "ŋ",
        "kh": "χ",
        "q": "k",
        # for foreign characters
        "w": "w",
        "f": "f",
        "z": "z",
        "j": "j",
        "p": "p"
    }

    medial2ipa = {
        None: None,
        # "u": "u̯",
        # "o": "u̯"
        "u": "w",
        "o": "w"
    }

    nucleus2ipa = {
        None: None,
        "iê": "iə",
        "yê": "iə",
        "ia": "iə",
        "ya": "iə",
        "uô": "uə",
        "ua": "uə",
        "ươ": "ɯə",
        "ưa": "ɯə",
        "a": "a",
        "ă": "ă",
        "â": "ə̆",
        "i": "i",
        "y": "i",
        "e": "ɛ",
        "ê": "e",
        "u": "u",
        "ư": "ɯ",
        "o": "ɔ",
        "oo": "ɔː",
        "ô": "o",
        "ơ": "ə"
    }

    coda2ipa = {
        None: None,
        "i": "j",
        "y": "j",
        "m": "m",
        "n": "n",
        "nh": "ŋ̟",
        "ng": "ŋ",
        "p": "p",
        "t": "t",
        "c": "k",
        "k": "k",
        "ch": "k̟",
        "u": "w",
        "o": "w",
    }

    mark2ipa = {
        None: "-",
        "<`>": "˨˩",
        "<?>": "˧˩",
        "<~>": "˧ˀ˥",
        "</>": "˧˥",
        "<.>": "˧ˀ˩",
    }

    is_Vietnamese_word, components = is_Vietnamese(syllable)

    if not is_Vietnamese_word:
        return None

    onset, medial, nucleus, coda, mark = components

    initial = onset2ipa[onset]
    glide = medial2ipa[medial]
    vowel = nucleus2ipa[nucleus]
    final = coda2ipa[coda]
    tone = mark2ipa[mark]

    if vowel == "a" and final in ["j", "w"]:
        if coda in ["y", "u"]:
            vowel = "ă"

    return initial, glide, vowel, final, tone


def analyse_Vietnamese(syllable: str) -> tuple[str]:
    special_cases = {
        "ty": "ti",
        "tỳ": "tì",
        "hỷ": "hỉ",
        "lý": "lí",
        "kỳ": "kì",
        "ký": "kí",
        "tỷ": "tỉ",
        "mỹ": "mĩ",
        "kỷ": "kỉ",
        "lỵ": "lị",
        "mỵ": "mị",
        "tý": "tí"
    }
    if syllable in special_cases:
        syllable = special_cases[syllable]

    components = convert_Vietnamese_to_IPA(syllable)

    if components:
        initial, medial, nucleus, final, tone = components

        rhyme = ""
        if medial:
            rhyme += medial
        if nucleus:
            rhyme += nucleus
        if final:
            rhyme += final

        return initial, rhyme, tone

    return None


In [31]:
word = compose_word(initial=None, rhyme=None, tone='-')

AttributeError: 'NoneType' object has no attribute 'startswith'

In [26]:
word

'tiếnh'

In [32]:
text = "đạt"

In [33]:
components = analyse_Vietnamese(text)

In [34]:
components

('d', 'at', '˧ˀ˩')

In [35]:
phonemes = set()

In [36]:
if components:
    phonemes.update([phoneme for phoneme in components if phoneme])

In [37]:
phonemes

{'at', 'd', '˧ˀ˩'}

In [38]:
specials = [0,1,2,3]

In [39]:
specials = [0]

In [40]:
specials

[0]

In [56]:
import torch
import torch.nn as nn

d_model = 8
embed_dim = d_model * 3
num_heads = 4

self_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

# input
batch_size = 2
seq_len = 5

x = torch.randn(batch_size, seq_len, embed_dim)

# causal mask
attn_mask = torch.triu(
    torch.ones(seq_len, seq_len) * float("-inf"),
    diagonal=1
)

out, weights = self_attn(
    x, x, x,
    attn_mask=attn_mask
)

print("input:", x.shape)
print("output:", out.shape)
print("attn weights:", weights.shape)

input: torch.Size([2, 5, 24])
output: torch.Size([2, 5, 24])
attn weights: torch.Size([2, 5, 5])


# Padding mask

In [18]:
def create_padding_mask(seq, pad_idx):
    # Input: Torch(B, S, 3)
    """
    Tạo mask cho key_padding_mask (True là Pad).
    Shape: (Batch_Size, Seq_Len)
    """
    seq = seq[:, :, 0]
    return (seq == pad_idx)

In [19]:
import torch

x = torch.tensor([
    [[1.0, 0.0, 2.0],
     [0.5, 1.0, 0.3],
     [0.2, 0.1, 0.4],
     [0.0, 0.0, 0.0]],   # PAD

    [[1.1, 0.2, 0.3],
     [0.4, 0.5, 0.6],
     [0.7, 0.8, 0.9],
     [0.0, 0.0, 0.0]]    # PAD
])

In [20]:
x.shape

torch.Size([2, 4, 3])

In [21]:
x[:, :, 0].shape

torch.Size([2, 4])

In [23]:
result = create_padding_mask(x, 0)

In [26]:
result.shape

torch.Size([2, 4])

# Causual mask


In [29]:
import torch
def create_causal_mask(seq_len):
        """
        Tạo mask Causal dạng Boolean để khớp với padding_mask.
        Logic: True = Che (Ignore), False = Nhìn (Keep).
        """
        # Tạo ma trận True ở tam giác trên (vị trí tương lai cần che)
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
        return mask

In [3]:
import torch

x = torch.tensor([
    [[1.0, 0.0, 2.0],
     [0.5, 1.0, 0.3],
     [0.2, 0.1, 0.4],
     [0.0, 0.0, 0.0]],   # PAD

    [[1.1, 0.2, 0.3],
     [0.4, 0.5, 0.6],
     [0.7, 0.8, 0.9],
     [0.0, 0.0, 0.0]]    # PAD
])

In [7]:
B, S, _ = x.shape

In [32]:
result = create_causal_mask(5)

In [33]:
result

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [34]:
result.device

device(type='cpu')

In [35]:
import torch
import math 
from torch import nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer (not a parameter, but part of state)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: [B, L, D*3]
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :].unsqueeze(0)

In [1]:
result

NameError: name 'result' is not defined

In [8]:
scores = [10, 5, 6, 7]
sum(scores)

28

In [9]:
from functools import lru_cache

In [14]:
from typing import Union
from functools import lru_cache

import torch
import os.path

def _get_invalid_locations_mask_fixed_dilation(seq_len: int, w: int, d: int):
    diagonals_list = []
    for j in range(-d * w, d, d):
        diagonal_mask = torch.zeros(seq_len, device='cpu', dtype=torch.uint8)
        diagonal_mask[:-j] = 1
        diagonals_list.append(diagonal_mask)
    return torch.stack(diagonals_list, dim=-1)

@lru_cache()
def _get_invalid_locations_mask(w: int, d: Union[torch.Tensor,int], autoregressive: bool, device: str):
    if isinstance(d, int):
        affected_seq_len = w * d
        mask = _get_invalid_locations_mask_fixed_dilation(affected_seq_len, w, d)
        mask = mask[None, :, None, :]
    else:
        affected_seq_len = w * d.max()
        head_masks = []
        d_list = d.cpu().numpy().tolist()
        for d in d_list:
            one_head_mask = _get_invalid_locations_mask_fixed_dilation(affected_seq_len, w, d)
            head_masks.append(one_head_mask)
        mask = torch.stack(head_masks, dim=-2)
        mask = mask[None, :, :, :]

    ending_mask = None if autoregressive else mask.flip(dims=(1, 3)).bool().to(device)
    return affected_seq_len, mask.bool().to(device), ending_mask

def mask_invalid_locations(input_tensor: torch.Tensor, w: int, d: Union[torch.Tensor, int], autoregressive: bool) -> torch.Tensor:
    affected_seq_len, beginning_mask, ending_mask = _get_invalid_locations_mask(w, d, autoregressive, input_tensor.device)
    seq_len = input_tensor.size(1)
    beginning_input = input_tensor[:, :affected_seq_len, :, :w+1]
    beginning_mask = beginning_mask[:, :seq_len].expand(beginning_input.size())
    beginning_input.masked_fill_(beginning_mask, -float('inf'))
    if not autoregressive:
        ending_input = input_tensor[:, -affected_seq_len:, :, -(w+1):]
        ending_mask = ending_mask[:, -seq_len:].expand(ending_input.size())
        ending_input.masked_fill_(ending_mask, -float('inf'))
    return input_tensor

# The non-tvm implementation is the default, we don't need to load the kernel at loading time.
# DiagonaledMM._get_function('float32', 'cuda')

In [15]:
import torch
from typing import Union
from functools import lru_cache

# Giả sử các hàm của bạn đã được định nghĩa ở trên...

def test_masking():
    # 1. Thiết lập tham số
    batch = 1
    seq_len = 10
    num_heads = 1
    w = 2  # window size (mỗi bên nhìn 2 token)
    d = 1  # dilation (không giãn cách)
    
    # 2. Tạo input_tensor giả lập
    # Trong các model như Longformer, input_tensor cho sliding window thường có 
    # chiều cuối là (2w + 1) hoặc tương tự. Ở đây hàm của bạn dùng w+1.
    # Giả sử shape là (Batch, Seq_len, Heads, Window_Width)
    input_tensor = torch.zeros((batch, seq_len, num_heads, w + 1), dtype=torch.float32)
    
    print(f"Original tensor shape: {input_tensor.shape}")

    output = mask_invalid_locations(input_tensor, w, d, autoregressive=False)
    
    print("\n--- Kết quả Masking (Hệ số w=2, d=1) ---")
    # In ra head đầu tiên, lát cắt của các vị trí mask
    # Giá trị -inf nghĩa là đã bị mask thành công
    print(output[0, :, 0, :]) 
    
    # 4. Kiểm tra điều kiện logic
    has_inf = torch.isinf(output).any()
    print(f"\nCó giá trị -inf (đã mask) không? {has_inf.item()}")
        

if __name__ == "__main__":
    test_masking()

Original tensor shape: torch.Size([1, 10, 1, 3])

--- Kết quả Masking (Hệ số w=2, d=1) ---
tensor([[-inf, -inf, 0.],
        [-inf, 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., -inf],
        [0., -inf, -inf]])

Có giá trị -inf (đã mask) không? True


In [19]:
import torch 
from torch import nn
import torch.nn.functional as F  
import math
from models.transformer_phoneme_longformer.utils.sliding_chunk import sliding_chunks_matmul_qk, sliding_chunks_matmul_pv
from models.transformer_phoneme_longformer.utils.diagonaled_mm_tvm import diagonaled_mm as diagonaled_mm_tvm, mask_invalid_locations

class LongformerSelfAttention(nn.Module):
    def __init__(self, config, layer_id):
        super(LongformerSelfAttention, self).__init__()
        if config.d_model % config.num_attention_heads != 0:
            raise ValueError(
                "The hidden size (%d) is not a multiple of the number of attention "
                "heads (%d)" % (config.d_model, config.num_attention_heads))
        
        self.num_heads = config.num_attention_heads
        self.head_dim = int(config.d_model / config.num_attention_heads)
        self.embed_dim = config.d_model

        self.query = nn.Linear(config.d_model, self.embed_dim)
        self.key = nn.Linear(config.d_model, self.embed_dim)
        self.value = nn.Linear(config.d_model, self.embed_dim)

        self.query_global = nn.Linear(config.d_model, self.embed_dim)
        self.key_global = nn.Linear(config.d_model, self.embed_dim)
        self.value_global = nn.Linear(config.d_model, self.embed_dim)

        self.dropout = config.attention_probs_dropout_prob

        self.layer_id = layer_id
        self.attention_window = config.attention_window[self.layer_id]
        self.attention_dilation = config.attention_dilation[self.layer_id]
        
        self.attention_mode = config.attention_mode
        self.autoregressive = config.autoregressive
        
        assert self.attention_window > 0
        assert self.attention_dilation > 0
        assert self.attention_mode in ['tvm', 'sliding_chunks']
        
        if self.attention_mode == 'sliding_chunks':
            assert not self.autoregressive  # not supported for sliding_chunks
            assert self.attention_dilation == 1  # dilation is not supported

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=False,
    ):
        assert encoder_hidden_states is None, "`encoder_hidden_states` is not supported and should be None"
        assert encoder_attention_mask is None, "`encoder_attention_mask` is not supported and should be None"

        if attention_mask is not None:
            attention_mask = attention_mask.squeeze(dim=2).squeeze(dim=1)
            key_padding_mask = attention_mask < 0
            extra_attention_mask = attention_mask > 0
            remove_from_windowed_attention_mask = attention_mask != 0

            num_extra_indices_per_batch = extra_attention_mask.long().sum(dim=1)
            max_num_extra_indices_per_batch = num_extra_indices_per_batch.max()
            if max_num_extra_indices_per_batch <= 0:
                extra_attention_mask = None
            else:
                extra_attention_mask_nonzeros = extra_attention_mask.nonzero(as_tuple=True)
                zero_to_max_range = torch.arange(0, max_num_extra_indices_per_batch,
                                                 device=num_extra_indices_per_batch.device)
                selection_padding_mask = zero_to_max_range < num_extra_indices_per_batch.unsqueeze(dim=-1)
                selection_padding_mask_nonzeros = selection_padding_mask.nonzero(as_tuple=True)
                selection_padding_mask_zeros = (selection_padding_mask == 0).nonzero(as_tuple=True)
        else:
            remove_from_windowed_attention_mask = None
            extra_attention_mask = None
            key_padding_mask = None

        hidden_states = hidden_states.transpose(0, 1)
        seq_len, bsz, embed_dim = hidden_states.size()
        assert embed_dim == self.embed_dim
        
        q = self.query(hidden_states)
        k = self.key(hidden_states)
        v = self.value(hidden_states)
        q /= math.sqrt(self.head_dim)

        q = q.view(seq_len, bsz, self.num_heads, self.head_dim).transpose(0, 1)
        k = k.view(seq_len, bsz, self.num_heads, self.head_dim).transpose(0, 1)
        
        # Use selected attention mode
        if self.attention_mode == 'tvm':
            q = q.float().contiguous()
            k = k.float().contiguous()
            attn_weights = diagonaled_mm_tvm(q, k, self.attention_window, self.attention_dilation, False, 0, False)
        elif self.attention_mode == "sliding_chunks":
            attn_weights = sliding_chunks_matmul_qk(q, k, self.attention_window, padding_value=0)
        else:
            raise ValueError(f"Unsupported attention mode: {self.attention_mode}")
        
        mask_invalid_locations(attn_weights, self.attention_window, self.attention_dilation, False)
        
        if remove_from_windowed_attention_mask is not None:
            remove_from_windowed_attention_mask = remove_from_windowed_attention_mask.unsqueeze(dim=-1).unsqueeze(dim=-1)
            float_mask = remove_from_windowed_attention_mask.type_as(q).masked_fill(remove_from_windowed_attention_mask, -10000.0)
            repeat_size = 1 if isinstance(self.attention_dilation, int) else len(self.attention_dilation)
            float_mask = float_mask.repeat(1, 1, repeat_size, 1)
            ones = float_mask.new_ones(size=float_mask.size())
            if self.attention_mode == 'tvm':
                d_mask = diagonaled_mm_tvm(ones, float_mask, self.attention_window, self.attention_dilation, False, 0, False)
            elif self.attention_mode == "sliding_chunks":
                d_mask = sliding_chunks_matmul_qk(ones, float_mask, self.attention_window, padding_value=0)
            else:
                raise ValueError(f"Unsupported attention mode: {self.attention_mode}")
            attn_weights += d_mask

        assert list(attn_weights.size())[:3] == [bsz, seq_len, self.num_heads]

        # Handle extra attention (global attention)
        if extra_attention_mask is not None:
            selected_k = k.new_zeros(bsz, max_num_extra_indices_per_batch, self.num_heads, self.head_dim)
            selected_k[selection_padding_mask_nonzeros] = k[extra_attention_mask_nonzeros]
            selected_attn_weights = torch.einsum('blhd,bshd->blhs', (q, selected_k))
            selected_attn_weights[selection_padding_mask_zeros[0], :, :, selection_padding_mask_zeros[1]] = -10000
            attn_weights = torch.cat((selected_attn_weights, attn_weights), dim=-1)

        attn_weights_float = F.softmax(attn_weights, dim=-1, dtype=torch.float32)
        
        if key_padding_mask is not None:
            attn_weights_float = torch.masked_fill(attn_weights_float, key_padding_mask.unsqueeze(-1).unsqueeze(-1), 0.0)
        
        attn_weights = attn_weights_float.type_as(attn_weights)
        attn_probs = F.dropout(attn_weights_float.type_as(attn_weights), p=self.dropout, training=self.training)
        
        v = v.view(seq_len, bsz, self.num_heads, self.head_dim).transpose(0, 1)
        attn = 0
        
        if extra_attention_mask is not None:
            selected_attn_probs = attn_probs.narrow(-1, 0, max_num_extra_indices_per_batch)
            selected_v = v.new_zeros(bsz, max_num_extra_indices_per_batch, self.num_heads, self.head_dim)
            selected_v[selection_padding_mask_nonzeros] = v[extra_attention_mask_nonzeros]
            attn = torch.matmul(selected_attn_probs.transpose(1, 2), selected_v.transpose(1, 2).type_as(selected_attn_probs)).transpose(1, 2)
            attn_probs = attn_probs.narrow(-1, max_num_extra_indices_per_batch, attn_probs.size(-1) - max_num_extra_indices_per_batch).contiguous()

        if self.attention_mode == 'tvm':
            v = v.float().contiguous()
            attn += diagonaled_mm_tvm(attn_probs, v, self.attention_window, self.attention_dilation, True, 0, False)
        elif self.attention_mode == "sliding_chunks":
            attn += sliding_chunks_matmul_pv(attn_probs, v, self.attention_window)
        else:
            raise ValueError(f"Unsupported attention mode: {self.attention_mode}")
        
        attn = attn.type_as(hidden_states)
        assert list(attn.size()) == [bsz, seq_len, self.num_heads, self.head_dim]
        attn = attn.transpose(0, 1).reshape(seq_len, bsz, embed_dim).contiguous()

        # Handle global attention recomputation
        if extra_attention_mask is not None:
            selected_hidden_states = hidden_states.new_zeros(max_num_extra_indices_per_batch, bsz, embed_dim)
            selected_hidden_states[selection_padding_mask_nonzeros[::-1]] = hidden_states[extra_attention_mask_nonzeros[::-1]]

            q = self.query_global(selected_hidden_states)
            k = self.key_global(hidden_states)
            v = self.value_global(hidden_states)
            q /= math.sqrt(self.head_dim)

            q = q.contiguous().view(max_num_extra_indices_per_batch, bsz * self.num_heads, self.head_dim).transpose(0, 1)
            k = k.contiguous().view(-1, bsz * self.num_heads, self.head_dim).transpose(0, 1)
            v = v.contiguous().view(-1, bsz * self.num_heads, self.head_dim).transpose(0, 1)
            
            attn_weights = torch.bmm(q, k.transpose(1, 2))
            assert list(attn_weights.size()) == [bsz * self.num_heads, max_num_extra_indices_per_batch, seq_len]

            attn_weights = attn_weights.view(bsz, self.num_heads, max_num_extra_indices_per_batch, seq_len)
            attn_weights[selection_padding_mask_zeros[0], :, selection_padding_mask_zeros[1], :] = -10000.0
            
            if key_padding_mask is not None:
                attn_weights = attn_weights.masked_fill(
                    key_padding_mask.unsqueeze(1).unsqueeze(2),
                    -10000.0,
                )
            
            attn_weights = attn_weights.view(bsz * self.num_heads, max_num_extra_indices_per_batch, seq_len)
            attn_weights_float = F.softmax(attn_weights, dim=-1, dtype=torch.float32)
            attn_probs = F.dropout(attn_weights_float.type_as(attn_weights), p=self.dropout, training=self.training)
            selected_attn = torch.bmm(attn_probs, v)
            
            assert list(selected_attn.size()) == [bsz * self.num_heads, max_num_extra_indices_per_batch, self.head_dim]

            selected_attn_4d = selected_attn.view(bsz, self.num_heads, max_num_extra_indices_per_batch, self.head_dim)
            nonzero_selected_attn = selected_attn_4d[selection_padding_mask_nonzeros[0], :, selection_padding_mask_nonzeros[1]]
            attn[extra_attention_mask_nonzeros[::-1]] = nonzero_selected_attn.view(len(selection_padding_mask_nonzeros[0]), -1).type_as(hidden_states)

        context_layer = attn.transpose(0, 1)
        
        if output_attentions:
            if extra_attention_mask is not None:
                attn_weights = attn_weights.view(bsz, self.num_heads, max_num_extra_indices_per_batch, seq_len)
            else:
                attn_weights = attn_weights.permute(0, 2, 1, 3)
        
        outputs = (context_layer, attn_weights) if output_attentions else (context_layer,)
        return outputs

ModuleNotFoundError: No module named 'models'

In [ ]:
import torch
import math

# Giả lập class Config để truyền vào model
class LongformerConfig:
    def __init__(self):
        self.d_model = 128
        self.num_attention_heads = 4
        self.attention_probs_dropout_prob = 0.1
        # Longformer yêu cầu định nghĩa cửa sổ cho từng layer
        self.attention_window = [32] * 12  # Cửa sổ 32 cho 12 layers
        self.attention_dilation = [1] * 12  # Dilation 1 cho 12 layers
        self.attention_mode = 'sliding_chunks' # 'tvm' hoặc 'sliding_chunks'
        self.autoregressive = False

def test_longformer_attention():
    # 1. Khởi tạo config và layer
    config = LongformerConfig()
    layer_id = 0
    longformer_attn = LongformerSelfAttention(config, layer_id)
    longformer_attn.eval() # Chuyển sang chế độ eval để tắt dropout

    # 2. Tạo dữ liệu giả lập (Sequence Length, Batch Size, Hidden Size)
    # Lưu ý: Longformer thường yêu cầu Seq_len là bội số của attention_window * 2
    # nếu dùng sliding_chunks.
    bsz = 2
    seq_len = 64 
    hidden_size = config.d_model
    
    # Hidden states trong code của bạn ban đầu được transpose(0,1) 
    # nên input đầu vào nên là (Seq_len, Batch, Hidden)
    hidden_states = torch.randn(seq_len, bsz, hidden_size)

    # 3. Tạo attention_mask giả lập
    # 0: local attention, 1: global attention, -1: padding
    # Ở đây chúng ta cho 2 token đầu tiên là global attention
    attention_mask = torch.zeros(bsz, seq_len)
    attention_mask[:, :2] = 1 
    # Thêm 4 chiều (bsz, 1, 1, seq_len) như Bert thường làm trước khi đưa vào layer
    # Nhưng code của bạn có đoạn .squeeze() nên ta đưa vào dạng (bsz, 1, 1, seq_len)
    attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)

    print(f"Input shape: {hidden_states.shape}")
    print(f"Attention mode: {config.attention_mode}")

    try:
        # 4. Chạy Forward
        outputs = longformer_attn(
            hidden_states,
            attention_mask=attention_mask,
            output_attentions=True
        )

        context_layer = outputs[0]
        print(f"\n--- Output thành công ---")
        print(f"Context layer shape: {context_layer.shape}") # Nên là (bsz, seq_len, hidden_size)
        
        if len(outputs) > 1:
            attn_weights = outputs[1]
            print(f"Attn weights shape: {attn_weights.shape}")

        # Kiểm tra tính toàn vẹn (không có NaN)
        if torch.isnan(context_layer).any():
            print("CẢNH BÁO: Output chứa NaN!")
        else:
            print("Kiểm tra dữ liệu: OK (không có NaN)")

    except Exception as e:
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    test_longformer_attention()